# Keyboard Hints Modal

> Modal-based keyboard shortcut reference with scannable grouped layout and `?` key trigger.

In [ ]:
#| default_exp components.hints_modal

In [ ]:
#| export
from __future__ import annotations
from typing import Optional
from fasthtml.common import Div, Span, Dialog, Button, Form, H3, Kbd, Script, FT

from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.hints import group_actions_by_hint_group

from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_backdrop
from cjm_fasthtml_daisyui.components.actions.button import btn_modifiers
from cjm_fasthtml_daisyui.components.data_display.kbd import kbd as kbd_cls, kbd_sizes as kbd_sz
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, border_dui

from cjm_fasthtml_design_system.text_tiers import text_tiers

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, items, gap, justify, flex_direction,
)
from cjm_fasthtml_tailwind.utilities.layout import position, right, top
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Design system recipes (V1 button roles, V11 icon-size roles)
from cjm_fasthtml_design_system.buttons import buttons
from cjm_fasthtml_design_system.icons import icons, IconSize

## Key Display

In [ ]:
#| export
def _render_key_combo(
    display_key: str,  # formatted key combo string (e.g., "Ctrl+Shift+\u2191")
) -> Div:              # container with kbd elements for each key part
    """Render a key combination as a sequence of kbd elements joined by `+`."""
    parts = display_key.split('+')
    elements = []
    for i, part in enumerate(parts):
        if i > 0:
            elements.append(Span('+', cls=combine_classes(text_tiers.muted, m.x(0.5))))
        elements.append(Kbd(part.strip(), cls=combine_classes(kbd_cls, kbd_sz.sm)))
    return Div(*elements, cls=combine_classes(flex_display, items.center))

In [ ]:
from fasthtml.common import to_xml

# Single key
html = to_xml(_render_key_combo("Space"))
assert "kbd" in html
assert "Space" in html
assert "+" not in html.split("kbd")[0]  # no plus before first kbd

# Multi-key combo
html = to_xml(_render_key_combo("Ctrl+Shift+\u2191"))
assert html.count("kbd") >= 3  # 3 kbd elements (tag appears in open+close)
assert "Ctrl" in html
assert "Shift" in html
assert "\u2191" in html
print("Key display tests passed")

Key display tests passed


## Hint Row & Group

In [ ]:
#| export
def _render_hint_row(
    display_key: str,  # formatted key combo string
    description: str,  # action description
) -> Div:              # single shortcut row with key and description
    """Render a single shortcut row: key combo on left, description on right."""
    return Div(
        _render_key_combo(display_key),
        Span(description, cls=combine_classes(text_tiers.secondary)),
        cls=combine_classes(
            flex_display, items.center, justify.between,
            gap(4), p.y(1),
        )
    )


def _render_modal_group(
    group_name: str,                        # group header text
    actions: list[tuple[str, str]],          # list of (display_key, description)
) -> Div:                                    # group container with header and rows
    """Render a group of related shortcuts with a header."""
    rows = [_render_hint_row(key, desc) for key, desc in actions]
    return Div(
        Div(
            group_name,
            cls=combine_classes(
                font_size.xs, font_weight.semibold,
                text_tiers.muted,
                p.b(1),
                border.b(), border_dui.base_content.opacity(10),
                m.b(1),
            )
        ),
        *rows,
        cls=m.b(4),
    )

In [ ]:
# Test hint row
row_html = to_xml(_render_hint_row("Ctrl+Z", "Undo last action"))
assert "Ctrl" in row_html
assert "Z" in row_html
assert "Undo last action" in row_html

# Test modal group
group_html = to_xml(_render_modal_group("Editing", [
    ("Enter", "Enter split mode"),
    ("Escape", "Exit split mode"),
]))
assert "Editing" in group_html
assert "Enter split mode" in group_html
assert "Exit split mode" in group_html
print("Hint row and group tests passed")

Hint row and group tests passed


## Modal Body

In [ ]:
#| export
def _render_modal_body(
    manager: ZoneManager,          # keyboard zone manager
    include_navigation: bool = True,  # include \u2191/\u2193 navigation hint
    include_zone_switch: bool = True, # include \u2190/\u2192 zone switch hint
) -> Div:                             # modal body with grouped shortcuts
    """Render the modal body with grouped keyboard shortcuts."""
    from cjm_fasthtml_keyboard_navigation.core.key_mapping import format_key_for_display

    groups_content = []

    # Navigation group (built-in)
    nav_hints = []
    if include_navigation:
        nav_hints.append(("\u2191 / \u2193", "Navigate items"))
    if include_zone_switch and len(manager.zones) > 1:
        prev_key = format_key_for_display(manager.prev_zone_key)
        next_key = format_key_for_display(manager.next_zone_key)
        nav_hints.append((f"{prev_key} / {next_key}", "Switch panel"))
    if nav_hints:
        groups_content.append(_render_modal_group("Navigation", nav_hints))

    # Action groups (from KeyAction hint_group)
    action_groups = group_actions_by_hint_group(manager.actions)
    for group_name, group_actions in action_groups.items():
        hints = [(a.get_display_key(), a.description) for a in group_actions]
        groups_content.append(_render_modal_group(group_name, hints))

    return Div(*groups_content, cls=p.t(2))

In [ ]:
from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager

# Build a test manager with two zones and some actions
z1 = FocusZone(id="seg")
z2 = FocusZone(id="align")
test_manager = ZoneManager(
    zones=(z1, z2),
    actions=(
        KeyAction(key="Enter", js_callback="x", description="Enter split mode", hint_group="Editing"),
        KeyAction(key="Escape", js_callback="x", description="Exit split mode", hint_group="Editing"),
        KeyAction(key="Backspace", htmx_trigger="x", description="Merge with previous", hint_group="Editing"),
        KeyAction(key="z", modifiers=frozenset({"ctrl"}), htmx_trigger="x", description="Undo", hint_group="Editing"),
        KeyAction(key=" ", js_callback="x", description="Play audio", hint_group="Audio"),
    ),
    prev_zone_key="ArrowLeft",
    next_zone_key="ArrowRight",
)

body_html = to_xml(_render_modal_body(test_manager))
assert "Navigation" in body_html
assert "Navigate items" in body_html
assert "Switch panel" in body_html
assert "Editing" in body_html
assert "Enter split mode" in body_html
assert "Audio" in body_html
assert "Play audio" in body_html
print("Modal body tests passed")

Modal body tests passed


## Trigger Button

In [ ]:
#| export
def render_keyboard_hints_trigger(
    modal_id: str = "kb-hints-modal",             # ID of the modal dialog to open
    icon_size: IconSize = icons.ghost_button,     # lucide icon size (V11.R3 ghost-button: "full" — pairs with V1.modal_disclosure at btn-xs)
) -> Button:                                      # ghost button with keyboard icon
    """Render a keyboard icon button that opens the hints modal."""
    return Button(
        lucide_icon("keyboard", size=icon_size),
        cls=combine_classes(buttons.modal_disclosure, btn_modifiers.circle),
        title="Keyboard shortcuts (?)",
        onclick=f"document.getElementById('{modal_id}').showModal();",
        type="button",
    )

In [ ]:
trigger = render_keyboard_hints_trigger()
html = to_xml(trigger)
assert "keyboard" in html.lower() or "svg" in html  # has icon
assert "showModal" in html
assert "kb-hints-modal" in html
assert 'title="Keyboard shortcuts (?)"' in html
print("Trigger button tests passed")

Trigger button tests passed


## Question Mark Key Listener

In [ ]:
#| export
def _render_question_mark_listener(
    modal_id: str,  # ID of the modal dialog to toggle
) -> Script:        # script element with global `?` key listener
    """Render a global `?` key listener that toggles the hints modal.
    
    Uses a named function stored on `window` so that HTMX re-renders
    replace the previous listener instead of accumulating duplicates.
    """
    return Script(f"""
    (function() {{
        // Remove previous listener if it exists (HTMX re-render dedup)
        if (window._kbHintsKeyListener) {{
            document.removeEventListener('keydown', window._kbHintsKeyListener);
        }}
        window._kbHintsKeyListener = function(e) {{
            // Skip if typing in an input, textarea, or contenteditable
            var tag = e.target.tagName;
            if (tag === 'INPUT' || tag === 'TEXTAREA' || e.target.isContentEditable) return;
            if (e.key === '?') {{
                e.preventDefault();
                var m = document.getElementById('{modal_id}');
                if (m) {{
                    if (m.open) {{ m.close(); }}
                    else {{ m.showModal(); }}
                }}
            }}
        }};
        document.addEventListener('keydown', window._kbHintsKeyListener);
    }})();
    """)


In [ ]:
listener = _render_question_mark_listener("kb-hints-modal")
html = to_xml(listener)
assert "keydown" in html
assert "e.key === '?'" in html
assert "showModal" in html
assert "INPUT" in html  # skips input fields
assert "TEXTAREA" in html
assert "isContentEditable" in html
print("Question mark listener tests passed")

Question mark listener tests passed


## Full Modal Component

In [ ]:
#| export
def render_keyboard_hints_modal(
    manager: ZoneManager,               # keyboard zone manager with actions configured
    modal_id: str = "kb-hints-modal",    # HTML ID for the modal dialog
    include_navigation: bool = True,     # include ↑/↓ navigation hint
    include_zone_switch: bool = True,    # include zone switch hint (auto-hidden for single zone)
    enable_question_mark_key: bool = True,  # add global `?` key listener
    title: str = "Keyboard Shortcuts",   # modal title text
) -> tuple[FT, FT, FT]:                 # (modal_dialog, trigger_button, question_mark_script)
    """Render a modal-based keyboard shortcut reference.

    Returns three components:
    - `modal_dialog`: The Dialog element (place anywhere in page)
    - `trigger_button`: Small keyboard icon button (place in step header)
    - `question_mark_script`: Global `?` key listener Script (place in page)

    If `enable_question_mark_key` is False, `question_mark_script` is an empty Div.
    """
    body = _render_modal_body(
        manager,
        include_navigation=include_navigation,
        include_zone_switch=include_zone_switch,
    )

    modal_dialog = Dialog(
        Div(
            # Close button (top-right corner)
            Form(
                Button(
                    "✕",
                    cls=combine_classes(
                        buttons.soft_dismissal, btn_modifiers.circle,
                        position.absolute, right._2, top._2,
                    ),
                ),
                method="dialog",
            ),
            # Title
            H3(
                lucide_icon("keyboard", size=icons.section_header, cls=str(m.r(2))),
                title,
                cls=combine_classes(
                    font_size.lg, font_weight.bold,
                    flex_display, items.center,
                ),
            ),
            # Shortcut groups
            body,
            # Footer hint
            Div(
                Span("Press "),
                Kbd("?", cls=combine_classes(kbd_cls, kbd_sz.sm)),
                Span(" to toggle this dialog"),
                cls=combine_classes(
                    font_size.xs, text_tiers.subtle,
                    p.t(3), border.t(), border_dui.base_content.opacity(10),
                    flex_display, items.center, gap(1),
                ),
            ),
            cls=combine_classes(modal_box, max_w.md),
        ),
        # Backdrop (click outside to close)
        Form(Button("close"), method="dialog", cls=str(modal_backdrop)),
        id=modal_id,
        cls=str(modal),
    )

    trigger = render_keyboard_hints_trigger(modal_id=modal_id)

    question_mark_script = (
        _render_question_mark_listener(modal_id)
        if enable_question_mark_key
        else Div(style="display:none;")
    )

    return modal_dialog, trigger, question_mark_script

In [ ]:
# Test with the dual-zone manager from above
modal_dialog, trigger, qm_script = render_keyboard_hints_modal(test_manager)

modal_html = to_xml(modal_dialog)
trigger_html = to_xml(trigger)
script_html = to_xml(qm_script)

# Modal structure
assert 'id="kb-hints-modal"' in modal_html
assert 'modal-box' in modal_html
assert 'modal-backdrop' in modal_html
assert 'Keyboard Shortcuts' in modal_html
assert 'Navigation' in modal_html
assert 'Editing' in modal_html
assert 'Audio' in modal_html
assert '\u2715' in modal_html  # close button

# Trigger
assert 'showModal' in trigger_html
assert 'kb-hints-modal' in trigger_html

# Question mark listener
assert 'keydown' in script_html
assert "e.key === '?'" in script_html

# Test with question mark key disabled
_, _, no_qm = render_keyboard_hints_modal(test_manager, enable_question_mark_key=False)
no_qm_html = to_xml(no_qm)
assert 'keydown' not in no_qm_html  # no listener
assert 'display:none' in no_qm_html  # empty placeholder

# Test single-zone manager (no zone switch hint)
single_manager = ZoneManager(
    zones=(z1,),
    actions=(KeyAction(key=" ", js_callback="x", description="Select", hint_group="Actions"),),
)
single_modal, _, _ = render_keyboard_hints_modal(single_manager)
single_html = to_xml(single_modal)
assert 'Switch panel' not in single_html  # no zone switch for single zone
assert 'Navigate items' in single_html
assert 'Select' in single_html

print("Full modal component tests passed")

Full modal component tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()